# Principle of least action: finding the path by gradient descent

*mechanics · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/mechanics/least_action)

Chop a path into points $q_n$ with fixed ends, write the action $S = \sum_n L\,\Delta t$ as one tensor expression and let autograd and an optimiser slide a batch of wrong guesses downhill until $\partial S/\partial q_n = 0$. They settle on the parabola of a thrown ball or the sine of an oscillator.

**Vector form**

$$S[q] = \int_0^T \Big(\tfrac12 m\dot q^2 - V(q)\Big)\,dt,\qquad \frac{\delta S}{\delta q(t)} = 0 \;\Leftrightarrow\; m\ddot q = -V'(q)$$

**Index (tensor) form**

$$S_b = \sum_{n=0}^{N-1}\Big[\tfrac12 m\,v_{bn}v_{bn} - V(q_{bn})\Big]\Delta t,\quad v_{bn} = \frac{D_{nk}\,q_{bk}}{\Delta t},\qquad \frac{\partial S_b}{\partial q_{bn}} = -\Delta t\Big[m\,\frac{q_{b,n+1} - 2q_{bn} + q_{b,n-1}}{\Delta t^2} + V'(q_{bn})\Big]$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $q_{bn}$ | `q[b, n]` | (B, N+1) | trial path $b$ at time $t_n = n\,\Delta t$; $q_{b0}$ and $q_{bN}$ are fixed |
| $D_{nk}$ | `difference_matrix(N)` | (N, N+1) | $-1$ on the diagonal, $+1$ right of it: $(Dq)_n = q_{n+1} - q_n$ |
| $v_{bn} = D_{nk} q_{bk}/\Delta t$ | `einsum('nk,bk->bn', D, q) / dt` | (B, N) | repeated $k$ ⇒ summed: velocity on every step of every path |
| $S_b$ | `action(q, D, dt, m, V)` | (B,) | kinetic minus potential, summed over the steps; $V$ taken at the left end of each step |
| $\partial S_b/\partial q_{bn}$ | `autograd.grad(S.sum(), x)` | (B, N-1) | gradient for the free interior points; the paths are independent so one backward pass serves all |
| $x \leftarrow x - \eta\,\partial S/\partial x$ | `optim.SGD([x], lr, momentum=0.9)` | (B, N-1) | gradient descent with momentum, $\eta = 0.8\,\Delta t/m$ |
| $\partial^2 S_b/\partial q_{bn}\partial q_{bm}$ | `einsum('bnbm->bnm', hessian(S))` | (B, N-1, N-1) | Hessian: tridiagonal, $2m/\Delta t - V''\Delta t$ on the diagonal and $-m/\Delta t$ beside it |
| $r_{bn}$ | `euler_lagrange_residual(q, dt, m, dV)` | (B, N-1) | discrete Euler–Lagrange equation $m(q_{n+1} - 2q_n + q_{n-1})/\Delta t^2 + V'(q_n)$, the Verlet scheme; $\partial S/\partial q_n = -\Delta t\,r_n$ |

Nature does not integrate forward step by step here: the whole path is adjusted at once, knowing where it must start and end. Setting $\partial S/\partial q_n = 0$ gives $m(q_{n+1} - 2q_n + q_{n-1})/\Delta t^2 = -V'(q_n)$, the discrete Euler–Lagrange equation, which is exactly the Verlet scheme; for constant gravity it reproduces the parabola to rounding error, for the oscillator to $O(\Delta t^2)$. The principle is really one of stationary action: for the oscillator the Hessian's lowest eigenvalue is about $m\Delta t\,(\pi^2/T^2 - \omega^2)$, so once $T$ exceeds half a period $\pi/\omega$ the true path is a saddle of $S$, gradient descent would run away, and the code switches to Newton steps $x \leftarrow x - \alpha H^{-1}\nabla S$, which find any stationary point. Smooth, long wiggles relax slowest (the smallest Hessian eigenvalues); short wiggles vanish within a few iterations.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/mechanics/least_action/main.py
"""Main loop: parameters → discrete action → wrong trial paths → minimise S → compare with the true path → plot."""

import math
from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    m, g, omega = 1.0, 9.81, params["omega"]
    system, T, N = params["system"], params["T"], params["n_steps"]
    q_start, q_end = params["q_start"], params["q_end"]
    V, dV = equation.gravity(m, g) if system == "gravity" else equation.oscillator(m, omega)
    dt = T / N

    t = tensors.time_grid(T, N, dev)  # (N+1,)
    D = equation.difference_matrix(N, device=dev)  # (N, N+1)
    q0 = tensors.trial_paths(t, q_start, q_end, params["n_paths"], params["wiggle"])  # (B, N+1)
    exact = equation.exact_path(t, q_start, q_end, system, g, omega)  # (N+1,)
    S_exact = equation.action(exact[None], D, dt, m, V)[0]

    # Is the true path a minimum? Look at the smallest eigenvalue of the (tridiagonal) Hessian.
    H = solve.hessian(q0[:1], D, dt, m, V)[0]  # (N−1, N−1)
    lowest = torch.linalg.eigvalsh(H)[0].item()
    band = H.triu(2).abs().max().item()
    print(
        f"Hessian {tuple(H.shape)}: diagonal {H[0, 0]:.3f}, off-diagonal {H[0, 1]:.3f}, "
        f"beyond the tridiagonal band {band:.1e}; smallest eigenvalue {lowest:.4f}"
    )
    method = params["method"]
    if lowest <= 0:
        print(f"T = {T} s is longer than half a period π/ω = {math.pi / omega:.3f} s: the true path is a saddle of S,")
        print("not a minimum, so gradient descent would run away. Switching to Newton steps (stationary action).")
        method = "newton"

    hist = solve.relax(q0, D, dt, m, V, params["n_iter"], method)
    q_final = hist["paths"][-1]  # (B, N+1)
    resid = equation.euler_lagrange_residual(q_final, dt, m, dV)  # (B, N−1)
    print(f"{method}: {params['n_iter']} iterations, {q0.shape[0]} trial paths in one tensor")
    print(
        f"action of the true path S = {S_exact:.6f} J·s, after relaxing S_b = "
        + ", ".join(f"{s:.6f}" for s in hist["S"][-1].tolist())
    )
    print(
        f"max |q_b − q_exact| = {(q_final - exact).abs().max():.2e} m,  "
        f"max |∂S/∂q_n| = {hist['grad'][-1].abs().max():.2e},  "
        f"max |discrete Euler–Lagrange residual| = {resid.abs().max():.2e}"
    )
    return plot.animate(t, hist, exact, S_exact, out_dir / "least_action.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/mechanics/least_action/equation.py
r"""Physics → tensor translation.

A path is a list of positions q_n = q(nΔt), n = 0 … N, with fixed ends q_0, q_N and Δt = T/N. The action

    S = \sum_{n=0}^{N-1} \Big[ \tfrac12 m \Big(\frac{q_{n+1} - q_n}{\Delta t}\Big)^2 - V(q_n) \Big] \Delta t

is the time integral of L = T − V with a forward difference for the velocity and the potential at the left end
of each step. Written with the difference matrix D_{nk} (D_{n,n} = −1, D_{n,n+1} = +1) the velocity is
v_{bn} = D_{nk} q_{bk} / Δt (repeated k ⇒ summed; b labels a path in the batch).

Setting ∂S/∂q_n = 0 for an interior point gives the discrete Euler–Lagrange equation

    \frac{\partial S}{\partial q_n} = -\Delta t \Big[ m\frac{q_{n+1} - 2q_n + q_{n-1}}{\Delta t^2} + V'(q_n) \Big] = 0,

which is exactly the Verlet scheme for m\ddot q = −V'(q). q_n appears only in its own term and its two
neighbours', so the Hessian ∂²S/∂q_n∂q_m is tridiagonal:  2m/Δt − V''Δt on the diagonal, −m/Δt next to it.
"""

from typing import Callable

import torch

Potential = Callable[[torch.Tensor], torch.Tensor]  # V(q) elementwise


def gravity(m: float, g: float) -> tuple[Potential, Potential]:
    """Ball thrown straight up, q = height:  V = m g q,  V' = m g."""
    return (lambda q: m * g * q), (lambda q: m * g * torch.ones_like(q))


def oscillator(m: float, omega: float) -> tuple[Potential, Potential]:
    """Mass on a spring:  V = ½ m ω² q²,  V' = m ω² q."""
    return (lambda q: 0.5 * m * omega**2 * q**2), (lambda q: m * omega**2 * q)


def difference_matrix(n_steps: int, dtype=torch.float64, device=None) -> torch.Tensor:
    """D of shape (N, N+1): (D q)_n = q_{n+1} − q_n."""
    eye = torch.eye(n_steps + 1, dtype=dtype, device=device)
    return eye[1:] - eye[:-1]


def action(q: torch.Tensor, D: torch.Tensor, dt: float, m: float, V: Potential) -> torch.Tensor:
    r"""S_b = Σ_n [½ m v_{bn} v_{bn} − V(q_{bn})] Δt  with  v_{bn} = D_{nk} q_{bk} / Δt.   (B, N+1) → (B,)"""
    v = torch.einsum("nk,bk->bn", D, q) / dt  # (B, N) velocity on each step, k summed
    kinetic = 0.5 * m * torch.einsum("bn,bn->b", v, v)  # n summed
    potential = V(q[:, :-1]).sum(dim=1)  # left end of every step, n = 0 … N−1
    return (kinetic - potential) * dt


def euler_lagrange_residual(q: torch.Tensor, dt: float, m: float, dV: Potential) -> torch.Tensor:
    """r_n = m (q_{n+1} − 2q_n + q_{n−1})/Δt² + V'(q_n) at the interior points.   (B, N+1) → (B, N−1)

    Zero for the true discrete path; ∂S/∂q_n = −Δt r_n.
    """
    accel = (q[:, 2:] - 2 * q[:, 1:-1] + q[:, :-2]) / dt**2
    return m * accel + dV(q[:, 1:-1])


def exact_path(t: torch.Tensor, q_start: float, q_end: float, system: str, g: float, omega: float) -> torch.Tensor:
    """The classical path through both end points.   (N+1,) → (N+1,)

    gravity:     q(t) = q_0 + (q_N − q_0) t/T + ½ g t (T − t)                    (a parabola)
    oscillator:  q(t) = [q_0 sin ω(T − t) + q_N sin ωt] / sin ωT                   (a sine)
    """
    T = t[-1]
    if system == "gravity":
        return q_start + (q_end - q_start) * t / T + 0.5 * g * t * (T - t)
    return (q_start * torch.sin(omega * (T - t)) + q_end * torch.sin(omega * t)) / torch.sin(omega * T)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/mechanics/least_action/tensors.py
"""Build the concrete tensors: the time grid and a batch of deliberately wrong trial paths.

float64 so the action can be pushed down to its minimum without rounding noise.
"""

import torch

DTYPE = torch.float64


def time_grid(T: float, n_steps: int, device=None) -> torch.Tensor:
    """t_n = n Δt for n = 0 … N: (N+1,)."""
    return torch.linspace(0.0, T, n_steps + 1, dtype=DTYPE, device=device)


def trial_paths(t: torch.Tensor, q_start: float, q_end: float, n_paths: int, scale: float) -> torch.Tensor:
    """B wrong guesses with the right end points: (B, N+1).

    Path 0 is the straight line between the ends; path b > 0 adds a wiggle a_b sin(k_b π t/T) that vanishes at
    both ends, with k_b = 1, 2, 3, … and alternating sign, so every guess starts in a different place.
    """
    T = t[-1]
    line = q_start + (q_end - q_start) * t / T  # (N+1,)
    b = torch.arange(n_paths, dtype=t.dtype, device=t.device)  # (B,)
    amp = scale * torch.where(b % 2 == 1, 1.0, -1.0) * (b > 0)  # (B,) path 0 has no wiggle
    wiggle = torch.sin(torch.outer(b, torch.pi * t / T))  # (B, N+1)  k_b = b
    return line + amp[:, None] * wiggle


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/mechanics/least_action/solve.py
"""Minimise the discrete action over the interior points: gradient descent with momentum, or damped Newton.

Only q_1 … q_{N−1} are free; the end points are glued back on before every evaluation of S.
"""

import torch

from problems._common.progress import steps

from .equation import Potential, action


def with_ends(x: torch.Tensor, q_start: torch.Tensor, q_end: torch.Tensor) -> torch.Tensor:
    """Interior points (B, N−1) → whole paths (B, N+1) with the fixed ends attached."""
    return torch.cat([q_start.expand(x.shape[0], 1), x, q_end.expand(x.shape[0], 1)], dim=1)


def hessian(q: torch.Tensor, D: torch.Tensor, dt: float, m: float, V: Potential) -> torch.Tensor:
    """∂²S_b/∂q_{bn}∂q_{bm} for every path by autograd: (B, N+1) → (B, N−1, N−1).

    The paths are independent, so the Hessian of ΣS_b is block diagonal in b: einsum 'bnbm->bnm' keeps the
    diagonal blocks (no index summed, b is just read twice).
    """
    ends = q[:, :1], q[:, -1:]

    def total(x):
        return action(torch.cat([ends[0], x, ends[1]], dim=1), D, dt, m, V).sum()

    full = torch.autograd.functional.hessian(total, q[:, 1:-1].detach(), vectorize=True)  # (B, N−1, B, N−1)
    return torch.einsum("bnbm->bnm", full)


def relax(
    q0: torch.Tensor, D: torch.Tensor, dt: float, m: float, V: Potential, n_iter: int, method: str
) -> dict[str, torch.Tensor]:
    """Push every trial path towards ∂S/∂q_n = 0. Returns the history of paths (K+1, B, N+1),
    actions (K+1, B) and gradients ∂S/∂q_n (K+1, B, N−1).

    gradient: x ← x − η ∂S/∂x + momentum, with η = 0.8 Δt/m (just below the stable limit set by the
              largest Hessian eigenvalue ≈ 4m/Δt). Works when S has a minimum (short T).
    newton:   x ← x − α H⁻¹ ∂S/∂x with α = 0.2. Finds any stationary point, also a saddle (long T).
    """
    q_start, q_end = q0[:, :1], q0[:, -1:]
    x = q0[:, 1:-1].clone().requires_grad_(True)  # (B, N−1) the free interior points
    optimiser = torch.optim.SGD([x], lr=0.8 * dt / m, momentum=0.9)
    paths, actions, grads = [], [], []
    for _ in steps(n_iter + 1, f"{method} iterations"):
        q = with_ends(x, q_start, q_end)  # (B, N+1)
        S = action(q, D, dt, m, V)  # (B,)
        (dS,) = torch.autograd.grad(S.sum(), x)  # (B, N−1): path b's own gradient (paths are independent)
        paths.append(q.detach().clone())
        actions.append(S.detach())
        grads.append(dS)
        if method == "newton":
            H = hessian(q.detach(), D, dt, m, V)  # (B, N−1, N−1) tridiagonal
            with torch.no_grad():
                x -= 0.2 * torch.linalg.solve(H, dS)
        else:
            optimiser.zero_grad()
            x.grad = dS
            optimiser.step()
    # n_iter + 1 states are stored: the start and the result of each of the n_iter updates
    return {"paths": torch.stack(paths), "S": torch.stack(actions), "grad": torch.stack(grads)}


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/mechanics/least_action/plot.py
"""Animate the trial paths relaxing onto the true path, with −∂S/∂q_n arrows and the action per iteration."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif


def animate(t: torch.Tensor, hist: dict, exact: torch.Tensor, S_exact: torch.Tensor, path: Path) -> Path:
    t = t.cpu().numpy()  # (N+1,)
    paths = hist["paths"].cpu().numpy()  # (K+1, B, N+1)
    S = hist["S"].cpu().numpy()  # (K+1, B)
    grad = hist["grad"].cpu().numpy()  # (K+1, B, N−1)
    exact = exact.cpu().numpy()
    S_exact = float(S_exact)
    n_iter, n_paths = S.shape[0] - 1, S.shape[1]
    iters = np.arange(n_iter + 1)

    fig, (ax, ax_s) = plt.subplots(1, 2, figsize=(8, 4), gridspec_kw={"width_ratios": [1.3, 1]})

    # --- left: the paths q(t) ---
    lo, hi = min(paths[0].min(), exact.min()), max(paths[0].max(), exact.max())
    pad = 0.08 * (hi - lo)
    ax.set_xlim(t[0], t[-1])
    ax.set_ylim(lo - pad, hi + pad)
    ax.set_xlabel("t (s)")
    ax.set_ylabel("q (m)")
    ax.plot(t, exact, color="#111827", lw=3, alpha=0.35, label="true path")
    lines = [ax.plot([], [], "-o", ms=2, lw=1.3, color=ACCENT[b % len(ACCENT)])[0] for b in range(n_paths)]
    ax.plot(t[[0, -1]], exact[[0, -1]], "o", ms=8, color="#374151", zorder=5)
    # arrows: the downhill direction −∂S/∂q_n on the last (most wiggly) path, scaled by the starting gradient
    arrow_scale = 0.12 * (hi - lo) / (np.abs(grad[0, -1]).max() + 1e-30)
    arrows = ax.quiver(
        t[1:-1],
        paths[0, -1, 1:-1],
        np.zeros(len(t) - 2),
        -grad[0, -1] * arrow_scale,
        angles="xy",
        scale_units="xy",
        scale=1,
        color=ACCENT[(n_paths - 1) % len(ACCENT)],
        width=0.004,
        alpha=0.7,
    )
    ax.legend(loc="lower center", fontsize=8)
    title = ax.set_title("iteration 0", fontsize=9)

    # --- right: the action of every path vs iteration ---
    excess = max(S[0].max() - S_exact, 1e-9)
    ax_s.set_xscale("log")
    ax_s.set_xlim(1, n_iter + 1)
    ax_s.set_ylim(S_exact - 0.08 * excess, S_exact + 1.05 * excess)
    ax_s.axhline(S_exact, color="#111827", lw=1.2, ls="--", label="S of the true path")
    s_lines = [ax_s.plot([], [], lw=1.3, color=ACCENT[b % len(ACCENT)])[0] for b in range(n_paths)]
    ax_s.set_xlabel("iteration + 1")
    ax_s.set_ylabel("action S (J·s)")
    ax_s.legend(loc="upper right", fontsize=8)

    def update(k: int):
        for b in range(n_paths):
            lines[b].set_data(t, paths[k, b])
            s_lines[b].set_data(iters[: k + 1] + 1, S[: k + 1, b])
        arrows.set_offsets(np.column_stack([t[1:-1], paths[k, -1, 1:-1]]))
        arrows.set_UVC(np.zeros(len(t) - 2), -grad[k, -1] * arrow_scale)
        title.set_text(f"iteration {k}:  max |∂S/∂q_n| = {np.abs(grad[k]).max():.1e}")

    # most of the change happens early: space the frames evenly in log(iteration)
    picks = frame_indices(n_iter + 1, 80)
    frames = [round((n_iter + 1) ** (p / max(1, n_iter))) - 1 for p in picks]
    fig.tight_layout()
    return save_gif(fig, update, frames, path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'system': 'gravity',
    'method': 'gradient',
    'T': 1.2,
    'omega': 2.0,
    'q_start': 0.0,
    'q_end': 1.0,
    'n_steps': 30,
    'n_paths': 4,
    'wiggle': 1.0,
    'n_iter': 400
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.mechanics.least_action")]:
    del sys.modules[name]
main = importlib.import_module("problems.mechanics.least_action.main")
problem_dir = Path("problems/mechanics/least_action")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
